# 01. Environment & Hardware Feasibility Audit
## LLM Inference Benchmark & Optimization Study

### Objectives
1. Inspect the physical hardware platform (CPU, RAM, Storage, GPU availability).
2. Establish strict memory budgets to prevent out-of-memory (OOM) crashes on 4 GB RAM systems.
3. Verify that the pre-flight safety hooks function properly prior to executing inference.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from src.inference_bench.config import load_local_config
from src.inference_bench.resource_monitor import ResourceMonitor

# 1. Load Local Configuration
cfg = load_local_config("../configs/local_cpu.yaml")
print(f"Target CPU Threads: {cfg.hardware.cpu_threads}")
print(f"Max Process Memory Budget: {cfg.hardware.max_memory_mb} MB")


In [ ]:
# 2. Inspect System Memory Status
monitor = ResourceMonitor(min_available_mb=400)
mem = monitor.get_system_memory()

print(f"Total Physical RAM: {mem['total_mb']} MB (~{mem['total_mb']/1024:.2f} GB)")
print(f"Available Physical RAM: {mem['available_mb']} MB (~{mem['available_mb']/1024:.2f} GB)")
print(f"Memory Utilization: {mem['used_percent']}%")


### Sizing & Memory Budget Formula
For an autoregressive decoder model with parameter count $P$ (in billions) and precision $b$ (bits per parameter):
$$\text{Active Model Memory} = P \times \frac{b}{8} \text{ GB}$$

For **Qwen2.5-0.5B**:
- **FP32 ($b=32$)**: $0.49 \times 4 = 1.96\text{ GB}$ (Exceeds available RAM)
- **FP16 ($b=16$)**: $0.49 \times 2 = 0.98\text{ GB}$ (Tight on 1.3 GB available RAM)
- **Q4_K_M ($b=4.5$)**: $0.49 \times 0.56 = 0.28\text{ GB}$ (~398 MB with tensor metadata, leaves > 850 MB free safety buffer).
